In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Stage 24: worked coding answers

Read each specification, predict a result and attempt an implementation before reading answers. Checks are examples, not exhaustive grading. These functions use NumPy; optional neural training is in the separate lab.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys
ROOT=Path.cwd()
while not (ROOT/'pyproject.toml').exists() and ROOT.parent!=ROOT:ROOT=ROOT.parent
if not (ROOT/'pyproject.toml').exists():raise RuntimeError('Open this notebook inside the extracted course.')
sys.path.insert(0,str(ROOT/'src'))
import numpy as np
import pytest

## 24.C1 — kick_drift

Advance the dimensionless oscillator with kick-drift symplectic Euler.

<details><summary>Hint</summary>Use the new momentum, not the old momentum, for position.</details>

In [3]:
def kick_drift(z, h):
    q,p=np.asarray(z,dtype=float)
    pn=p-h*q
    return np.array([q+h*pn,pn])

In [4]:
np.testing.assert_allclose(kick_drift([1,0],.2),[.96,-.2])
assert .5*np.sum(kick_drift([1,0],.2)**2)==pytest.approx(.4808)
print("24.C1: example checks passed; explain the assumptions.")

24.C1: example checks passed; explain the assumptions.


## 24.C2 — unit_energy

Compute H=(q²+p²)/2 along the final axis of an array.

<details><summary>Hint</summary>Preserve batch dimensions.</details>

In [5]:
def unit_energy(z):
    z=np.asarray(z,dtype=float)
    if z.ndim<1 or z.shape[-1]!=2 or not np.isfinite(z).all():raise ValueError('Finite (...,2) state required.')
    return .5*np.sum(z*z,axis=-1)

In [6]:
np.testing.assert_allclose(unit_energy([[1,0],[.96,-.2]]),[.5,.4808])
print("24.C2: example checks passed; explain the assumptions.")

24.C2: example checks passed; explain the assumptions.


## 24.C3 — linear_symplectic_error

Return Frobenius norm of MᵀJM-J in q-first, p-second ordering.

<details><summary>Hint</summary>Construct the block skew matrix; determinant alone is insufficient.</details>

In [7]:
def linear_symplectic_error(M):
    M=np.asarray(M,dtype=float)
    if M.ndim!=2 or M.shape[0]!=M.shape[1] or M.shape[0]%2 or not np.isfinite(M).all():raise ValueError('Even finite square map required.')
    d=M.shape[0]//2;I=np.eye(d);Z=np.zeros((d,d));J=np.block([[Z,I],[-I,Z]])
    return np.linalg.norm(M.T@J@M-J)

In [8]:
assert linear_symplectic_error([[.96,.2],[-.2,1.]])<1e-12
assert linear_symplectic_error(np.diag([2,.5,1,1]))>.1
print("24.C3: example checks passed; explain the assumptions.")

24.C3: example checks passed; explain the assumptions.


## Explain, do not just execute

Write one valid input, one invalid input, a hand calculation and a claim this function cannot establish. Record your evidence only after doing your own work.